In [2]:
import pandas as pd

df_data = pd.read_excel("C:/Users/bandarua/crda_gp/data/Concrete_Data.xls")
df_data.head()

,Cement (component 1)(kg in a m^3 mixture),Blast Furnace Slag (component 2)(kg in a m^3 mixture),Fly Ash (component 3)(kg in a m^3 mixture),Water (component 4)(kg in a m^3 mixture),Superplasticizer (component 5)(kg in a m^3 mixture),Coarse Aggregate (component 6)(kg in a m^3 mixture),Fine Aggregate (component 7)(kg in a m^3 mixture),Age (day),"Concrete compressive strength(MPa, megapascals)"
0,540.0,0.0,0.0,162.0,2.5,1040.0,676.0,28,79.986111
1,540.0,0.0,0.0,162.0,2.5,1055.0,676.0,28,61.887366
2,332.5,142.5,0.0,228.0,0.0,932.0,594.0,270,40.269535
3,332.5,142.5,0.0,228.0,0.0,932.0,594.0,365,41.052780
4,198.6,132.4,0.0,192.0,0.0,978.4,825.5,360,44.296075


In [ ]:
df_data["Concrete compressive strength(MPa, megapascals) "]

In [5]:
from sklearn.model_selection import train_test_split
import torch
from crda_gp.utils import set_seed
import numpy as np

set_seed(42)

X = np.array(df_data.drop(columns="Concrete compressive strength(MPa, megapascals) "))
y_1 = np.array(df_data["Concrete compressive strength(MPa, megapascals) "])

X_train, X_test, y_train, y_test = train_test_split(X, y_1, test_size=0.1, random_state=42)

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

In [6]:
import gpytorch
from crda_gp.models import ARDExactGPModel_Dense
set_seed(42)
likelihood = gpytorch.likelihoods.GaussianLikelihood()
model = ARDExactGPModel_Dense(train_x=X_train_tensor,
                              train_y=y_train_tensor,
                              likelihood=likelihood)
model

ARDExactGPModel_Dense(
  (likelihood): GaussianLikelihood(
    (noise_covar): HomoskedasticNoise(
      (raw_noise_constraint): GreaterThan(1.000E-04)
    )
  )
  (mean_module): ConstantMean()
  (covar_module): ScaleKernel(
    (base_kernel): RBFKernel(
      (raw_lengthscale_constraint): Positive()
    )
    (raw_outputscale_constraint): Positive()
  )
)

In [7]:
from crda_gp.utils import train_step
set_seed(42)
train_step(X_train=X_train_tensor,
           y_train=y_train_tensor,
           model=model,
           likelihood=likelihood)

Iter: 10/100 | Loss: 276.305 | Noise: 1.281
Iter: 20/100 | Loss: 174.164 | Noise: 1.893
Iter: 30/100 | Loss: 130.017 | Noise: 2.398
Iter: 40/100 | Loss: 107.275 | Noise: 2.789
Iter: 50/100 | Loss: 93.437 | Noise: 3.098
Iter: 60/100 | Loss: 83.849 | Noise: 3.354
Iter: 70/100 | Loss: 76.550 | Noise: 3.576
Iter: 80/100 | Loss: 70.674 | Noise: 3.775
Iter: 90/100 | Loss: 65.740 | Noise: 3.957
Iter: 100/100 | Loss: 61.473 | Noise: 4.125


In [8]:
from crda_gp.utils import val_step
set_seed(42)
y_pred = val_step(X_test=X_test_tensor,
                  model=model,
                  likelihood=likelihood)

In [9]:
from sklearn.metrics import mean_squared_error
set_seed(42)
MSE = mean_squared_error(y_true=y_test_tensor, 
                         y_pred=y_pred)

In [10]:
MSE


802.259765625

In [11]:
# Splitting the train data into augment and true values respectively
X_train_aug, X_train_true, y_train_aug, y_train_true = train_test_split(X_train, y_train, test_size=(0.6*len(X)/len(X_train)), random_state=42)
# Change the data into tensors
X_train_aug_tensor = torch.tensor(X_train_aug, dtype=torch.float32)
X_train_true_tensor = torch.tensor(X_train_true, dtype=torch.float32)
y_train_aug_tensor = torch.tensor(y_train_aug, dtype=torch.float32)
y_train_true_tensor = torch.tensor(y_train_true, dtype=torch.float32)

In [12]:
likelihood_true = gpytorch.likelihoods.GaussianLikelihood()
model_true = ARDExactGPModel_Dense(train_x=X_train_true_tensor,
                                   train_y=y_train_true_tensor,
                                   likelihood=likelihood_true).to(torch.float32)

In [13]:
train_step(X_train=X_train_true_tensor, 
           y_train=y_train_true_tensor,
           model=model_true,
           likelihood=likelihood_true)

Iter: 10/100 | Loss: 292.180 | Noise: 1.282
Iter: 20/100 | Loss: 184.778 | Noise: 1.895
Iter: 30/100 | Loss: 138.469 | Noise: 2.404
Iter: 40/100 | Loss: 114.457 | Noise: 2.800
Iter: 50/100 | Loss: 99.759 | Noise: 3.114
Iter: 60/100 | Loss: 89.530 | Noise: 3.374
Iter: 70/100 | Loss: 81.746 | Noise: 3.601
Iter: 80/100 | Loss: 75.455 | Noise: 3.804
Iter: 90/100 | Loss: 70.161 | Noise: 3.989
Iter: 100/100 | Loss: 65.574 | Noise: 4.161


In [14]:
y_pred_true = val_step(X_test=X_test_tensor, 
                       model=model_true, 
                       likelihood=likelihood_true)

In [15]:
MSE_true = mean_squared_error(y_test_tensor, y_pred_true)
MSE_true

915.226318359375

In [16]:
MSE

802.259765625

In [17]:
likelihood_aug = gpytorch.likelihoods.GaussianLikelihood()
model_aug = ARDExactGPModel_Dense(train_x=X_train_aug_tensor, 
                                  train_y=y_train_aug_tensor, 
                                  likelihood=likelihood_aug).to(torch.float32)

In [18]:
train_step(X_train=X_train_aug_tensor, 
           y_train=y_train_aug_tensor,
           model=model_aug,
           likelihood=likelihood_aug)

Iter: 10/100 | Loss: 288.241 | Noise: 1.282
Iter: 20/100 | Loss: 183.371 | Noise: 1.899
Iter: 30/100 | Loss: 138.026 | Noise: 2.413
Iter: 40/100 | Loss: 114.231 | Noise: 2.816
Iter: 50/100 | Loss: 99.615 | Noise: 3.136
Iter: 60/100 | Loss: 89.457 | Noise: 3.403
Iter: 70/100 | Loss: 81.759 | Noise: 3.635
Iter: 80/100 | Loss: 75.656 | Noise: 3.844
Iter: 90/100 | Loss: 70.402 | Noise: 4.035
Iter: 100/100 | Loss: 65.992 | Noise: 4.213


In [19]:
y_pred_aug = val_step(X_test=X_test_tensor, 
                      model=model_aug, 
                      likelihood=likelihood_aug)

In [21]:
MSE_aug = mean_squared_error(y_test_tensor, y_pred_aug)
MSE_aug

970.0380249023438

In [22]:
MSE

802.259765625

In [30]:
with torch.no_grad(), gpytorch.settings.fast_pred_var():
    pred_dist = likelihood_aug(model_aug(X_train_aug_tensor))
    y_new_aug = pred_dist.mean

residual = y_train_aug_tensor - y_new_aug
corr = abs(pd.DataFrame(X_train).corrwith(pd.Series(residual)))

preturb_idx= corr[abs(corr) < corr.mean().item()].index.values
preturb_idx

c:\Users\bandarua\crda_gp\env\Lib\site-packages\gpytorch\models\exact_gp.py:299: GPInputWarning: The input matches the stored training data. Did you forget to call model.train()?
  warnings.warn(


array([1, 2, 5])

In [31]:
from crda_gp.crda import CRDA_function
X_train_aug_CRDA_tensor, y_train_aug_CRDA_tensor, _, _ = CRDA_function(X_train_aug_tensor, 
                                                                 y_train_aug_tensor, 
                                                                 model=model_aug, 
                                                                 likelihood=likelihood_aug, 
                                                                 perturb_idx=preturb_idx, 
                                                                #  correlation=np.array(pd.DataFrame(X_train).corr())[preturb_idx],
                                                                 p=0.6)

c:\Users\bandarua\crda_gp\env\Lib\site-packages\gpytorch\models\exact_gp.py:299: GPInputWarning: The input matches the stored training data. Did you forget to call model.train()?
  warnings.warn(


Starting the Loop
Iter: 0
Iter: 1
Iter: 2
Iter: 3
Iter: 4
Iter: 5
Iter: 6
Iter: 7
Iter: 8
Iter: 9
Iter: 10
Iter: 11
Iter: 12
Iter: 13
Iter: 14
Iter: 15
Iter: 16
Iter: 17
Iter: 18
Iter: 19
Iter: 20
Iter: 21
Iter: 22
Iter: 23
Iter: 24
Iter: 25
Iter: 26
Iter: 27
Iter: 28
Iter: 29
Iter: 30
Iter: 31
Iter: 32
Iter: 33
Iter: 34
Iter: 35
Iter: 36
Iter: 37
Iter: 38
Iter: 39
Iter: 40
Iter: 41
Iter: 42
Iter: 43
Iter: 44
Iter: 45
Iter: 46
Iter: 47
Iter: 48
Iter: 49
Iter: 50
Iter: 51
Iter: 52
Iter: 53
Iter: 54
Iter: 55
Iter: 56
Iter: 57
Iter: 58
Iter: 59
Iter: 60
Iter: 61
Iter: 62
Iter: 63
Iter: 64
Iter: 65
Iter: 66
Iter: 67
Iter: 68
Iter: 69
Iter: 70
Iter: 71
Iter: 72
Iter: 73
Iter: 74
Iter: 75
Iter: 76
Iter: 77
Iter: 78
Iter: 79
Iter: 80
Iter: 81
Iter: 82
Iter: 83
Iter: 84
Iter: 85
Iter: 86
Iter: 87
Iter: 88
Iter: 89
Iter: 90
Iter: 91
Iter: 92
Iter: 93
Iter: 94
Iter: 95
Iter: 96
Iter: 97
Iter: 98
Iter: 99
Iter: 100
Iter: 101
Iter: 102
Iter: 103
Iter: 104
Iter: 105
Iter: 106
Iter: 107
Iter: 108
It

In [32]:
likelihood_aug_new = gpytorch.likelihoods.GaussianLikelihood()
model_aug_new = ARDExactGPModel_Dense(train_x=X_train_aug_CRDA_tensor, 
                                      train_y=y_train_aug_CRDA_tensor, 
                                      likelihood=likelihood_aug_new)

In [33]:
train_step(X_train=X_train_aug_CRDA_tensor, 
           y_train=y_train_aug_CRDA_tensor,
           model=model_aug_new,
           likelihood=likelihood_aug_new)

Iter: 10/100 | Loss: 189.585 | Noise: 1.282
Iter: 20/100 | Loss: 119.759 | Noise: 1.896
Iter: 30/100 | Loss: 89.636 | Noise: 2.406
Iter: 40/100 | Loss: 73.907 | Noise: 2.801
Iter: 50/100 | Loss: 64.257 | Noise: 3.115
Iter: 60/100 | Loss: 57.557 | Noise: 3.374
Iter: 70/100 | Loss: 52.472 | Noise: 3.599
Iter: 80/100 | Loss: 48.382 | Noise: 3.799
Iter: 90/100 | Loss: 44.812 | Noise: 3.983
Iter: 100/100 | Loss: 41.756 | Noise: 4.152


In [34]:
y_pred_aug_new = val_step(X_test=X_test_tensor, 
                      model=model_aug_new, 
                      likelihood=likelihood_aug_new)

In [35]:
MSE_aug_new = mean_squared_error(y_test_tensor, y_pred_aug_new)
MSE_aug_new

953.5526123046875

In [29]:
MSE_aug

970.0380249023438